# EDA ฝั่งฟีเจอร์ตัวเลข (Yanisa, wk1)

โน้ตบุ๊กนี้สำรวจฟีเจอร์ที่เป็น**ตัวเลข**ของ IBM HR dataset ว่ากลุ่มที่ลาออก (`Attrition = Yes`) ต่างจากกลุ่มที่อยู่ต่ออย่างไร เพื่อตอบว่าฟีเจอร์ไหนน่าจะช่วยทำนายการลาออกได้ และมีอะไรต้องระวังตอนเทรนโมเดล

การแบ่งงาน wk1 ตาม [TASKS.md](../TASKS.md): Yanisa ทำฝั่งตัวเลข (โน้ตบุ๊กนี้) ส่วน Nanthamon ทำฝั่งหมวดหมู่ (Department, JobRole, MaritalStatus, OverTime, BusinessTravel ฯลฯ) ในโน้ตบุ๊กของตัวเอง ไม่แก้ไฟล์เดียวกันจึงไม่ชนกัน

ใช้ข้อมูลดิบจาก `data/raw/` (ไม่ใช่ `data/processed/`) เพราะค่ายังอ่านง่าย ไม่โดน encode และโน้ตบุ๊กนี้ไม่แก้ข้อมูล แค่อ่านอย่างเดียว

ลำดับ:
1. โหลดข้อมูลและแบ่งกลุ่มฟีเจอร์ตัวเลข
2. ดูความไม่สมดุลของ target
3. เทียบค่ากลางระหว่างกลุ่มลาออก vs อยู่ต่อ + ทดสอบทางสถิติ
4. ดูการกระจายตัว (histogram / boxplot)
5. อัตราการลาออกแยกตามช่วงค่า (อ่านง่ายสุดสำหรับฝ่าย HR)
6. ฟีเจอร์แบบระดับ 1–4 (ordinal)
7. ความเบ้และ outlier
8. ความสัมพันธ์ระหว่างฟีเจอร์ (multicollinearity)
9. สรุปผล 3–5 ข้อ

---
## ขั้นตอนที่ 1: โหลดข้อมูลและแบ่งกลุ่มฟีเจอร์

ใช้ `load_raw_data` จาก [`src/clean_pipeline.py`](../src/clean_pipeline.py) ตัวเดียวกับที่ทีมใช้ ถ้าไม่มีไฟล์ในเครื่องจะดาวน์โหลดจาก Kaggle ให้ (ไฟล์อยู่ใน repo แล้ว ปกติไม่ต้องโหลด)

รูปที่สำคัญจะถูกบันทึกเป็น PNG ไว้ที่ `docs/figures/eda_numeric_Y/` เพื่อเอาไปใส่สไลด์ Data Gate (wk4) ได้เลย

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sys.path.append(os.path.abspath(".."))
from src.clean_pipeline import NOISE_COLUMNS, RAW_FILENAME, TARGET_COLUMN, load_raw_data

RAW_PATH = os.path.join("..", "data", "raw", RAW_FILENAME)
FIG_DIR = os.path.join("..", "docs", "figures", "eda_numeric_Y")
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")
PALETTE = {"Stayed": "#4C72B0", "Left": "#DD8452"}


def save(fig, name):
    """บันทึกรูปไว้ใช้ในสไลด์"""
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")


df = load_raw_data(RAW_PATH)
df["left"] = (df[TARGET_COLUMN] == "Yes").astype(int)
df["Group"] = df["left"].map({0: "Stayed", 1: "Left"})
df.shape  # ควรได้ (1470, 37) = 35 คอลัมน์เดิม + left + Group

ฟีเจอร์ตัวเลขแบ่งเป็น 2 แบบ เพราะวิธีดูต่างกัน:

- **ต่อเนื่อง/นับจำนวน** (อายุ เงินเดือน จำนวนปี ฯลฯ) ดูด้วย histogram, boxplot และแบ่งช่วง
- **ระดับ (ordinal)** เป็นคะแนน 1–4 หรือระดับขั้น (ความพึงพอใจ, JobLevel, StockOptionLevel) แม้เก็บเป็นตัวเลขแต่มีแค่ไม่กี่ค่า ดูเป็นกราฟแท่งอัตราลาออกต่อระดับจะชัดกว่า

ไม่รวม 4 คอลัมน์ noise ที่ทีมตัดทิ้งแล้วตอน cleaning (`EmployeeCount`, `StandardHours`, `EmployeeNumber` และ `Over18`)

In [ ]:
CONTINUOUS = [
    "Age", "DailyRate", "DistanceFromHome", "HourlyRate", "MonthlyIncome", "MonthlyRate",
    "NumCompaniesWorked", "PercentSalaryHike", "TotalWorkingYears", "TrainingTimesLastYear",
    "YearsAtCompany", "YearsInCurrentRole", "YearsSinceLastPromotion", "YearsWithCurrManager",
]
ORDINAL = [
    "Education", "EnvironmentSatisfaction", "JobInvolvement", "JobLevel", "JobSatisfaction",
    "PerformanceRating", "RelationshipSatisfaction", "StockOptionLevel", "WorkLifeBalance",
]
NUMERIC = CONTINUOUS + ORDINAL

# เช็คว่าไม่ตกหล่นคอลัมน์ตัวเลขไหน
all_numeric = set(df.select_dtypes("number").columns) - {"left"} - set(NOISE_COLUMNS)
assert all_numeric == set(NUMERIC), all_numeric ^ set(NUMERIC)
print(f"ต่อเนื่อง {len(CONTINUOUS)} คอลัมน์, ordinal {len(ORDINAL)} คอลัมน์")
print("missing values:", df[NUMERIC].isna().sum().sum())

---
## ขั้นตอนที่ 2: ความไม่สมดุลของ target

ถ้ากลุ่มลาออกมีน้อย การเทียบด้วยจำนวนคน (count) จะทำให้กลุ่มลาออกดูเล็กไปหมด กราฟต่อจากนี้จึงใช้ **density / อัตรา (%)** แทนจำนวนคน

In [ ]:
counts = df["Group"].value_counts()
overall_rate = df["left"].mean()
print(counts)
print(f"อัตราลาออกรวม: {overall_rate:.1%}")  # ควรได้ ~16.1% (237 จาก 1,470 คน)

---
## ขั้นตอนที่ 3: เทียบค่ากลาง ลาออก vs อยู่ต่อ

ตารางด้านล่างเทียบค่ามัธยฐาน (median) และค่าเฉลี่ยของสองกลุ่ม แล้วทดสอบด้วย **Mann-Whitney U** (ไม่ต้องสมมติว่าข้อมูลเป็น normal ซึ่งหลายคอลัมน์ไม่ใช่ ดูขั้นตอนที่ 7)

- `p_value` < 0.05 แปลว่าความต่างไม่น่าจะเกิดจากความบังเอิญ
- `effect` (rank-biserial) บอกว่าต่างกัน**มากแค่ไหน** อยู่ระหว่าง -1 ถึง 1 ค่าติดลบ = กลุ่มลาออกมีค่าต่ำกว่า เกณฑ์คร่าว ๆ: |effect| < 0.1 แทบไม่ต่าง, 0.1–0.3 ต่างเล็กน้อย, 0.3–0.5 ปานกลาง, > 0.5 มาก

เรียงจาก |effect| มากไปน้อย ฟีเจอร์บน ๆ คือตัวที่แยกสองกลุ่มได้ชัดที่สุด

In [ ]:
def compare_groups(data, cols):
    rows = []
    for col in cols:
        left = data.loc[data["left"] == 1, col]
        stayed = data.loc[data["left"] == 0, col]
        u, p = stats.mannwhitneyu(left, stayed, alternative="two-sided")
        rows.append({
            "feature": col,
            "median_left": left.median(),
            "median_stayed": stayed.median(),
            "mean_left": left.mean(),
            "mean_stayed": stayed.mean(),
            "p_value": p,
            "effect": 2 * u / (len(left) * len(stayed)) - 1,  # rank-biserial
        })
    out = pd.DataFrame(rows).set_index("feature")
    return out.reindex(out["effect"].abs().sort_values(ascending=False).index)


comparison = compare_groups(df, NUMERIC)
comparison.style.format({
    "median_left": "{:,.1f}", "median_stayed": "{:,.1f}",
    "mean_left": "{:,.1f}", "mean_stayed": "{:,.1f}",
    "p_value": "{:.2e}", "effect": "{:+.3f}",
}).background_gradient(subset=["effect"], cmap="RdBu", vmin=-0.4, vmax=0.4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
plot_df = comparison.iloc[::-1]
colors = ["#DD8452" if e > 0 else "#4C72B0" for e in plot_df["effect"]]
ax.barh(plot_df.index, plot_df["effect"], color=colors)
ax.axvline(0, color="black", lw=0.8)
for x in (-0.1, 0.1):
    ax.axvline(x, color="grey", ls=":", lw=0.8)
ax.set_xlabel("Effect size (rank-biserial)  <- lower in Left | higher in Left ->")
ax.set_title("How strongly each numeric feature separates leavers from stayers")
save(fig, "01_effect_size")
plt.show()

---
## ขั้นตอนที่ 4: การกระจายตัวของฟีเจอร์ต่อเนื่อง

histogram ใช้ `stat="density"` และ `common_norm=False` ให้พื้นที่ใต้กราฟของแต่ละกลุ่มเท่ากัน เทียบรูปทรงได้แม้กลุ่มลาออกมีคนน้อยกว่าเกือบ 5 เท่า

สังเกต: ถ้าสีส้ม (Left) เบ้ไปทางซ้ายของสีน้ำเงิน แปลว่ากลุ่มลาออกมีค่าต่ำกว่า

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 18))
for ax, col in zip(axes.flat, CONTINUOUS):
    sns.histplot(data=df, x=col, hue="Group", stat="density", common_norm=False,
                 kde=True, element="step", palette=PALETTE, ax=ax)
    ax.set_title(col)
for ax in axes.flat[len(CONTINUOUS):]:
    ax.set_visible(False)
fig.tight_layout()
save(fig, "02_histograms")
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 18))
for ax, col in zip(axes.flat, CONTINUOUS):
    sns.boxplot(data=df, x="Group", y=col, hue="Group", order=["Stayed", "Left"],
                palette=PALETTE, legend=False, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
for ax in axes.flat[len(CONTINUOUS):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

---
## ขั้นตอนที่ 5: อัตราลาออกแยกตามช่วงค่า

กราฟแบบนี้อ่านง่ายที่สุดสำหรับคนที่ไม่ใช่สาย data เช่น "พนักงานอายุต่ำกว่า 25 ปี ลาออกกี่ %" เหมาะเอาไปใส่สไลด์

ช่วง (bin) ของฟีเจอร์หลักตั้งเองให้อ่านเป็นภาษาคนได้ ส่วนฟีเจอร์อื่นแบ่งเป็น 5 ช่วงเท่า ๆ กันตามจำนวนคน (quantile) เส้นประคืออัตราลาออกรวม ~16% และตัวเลขบนแท่งคือจำนวนคนในช่วงนั้น (ช่วงที่มีคนน้อยให้ระวังการตีความ)

In [ ]:
CUSTOM_BINS = {
    "Age": [18, 25, 30, 35, 45, 61],
    "MonthlyIncome": [0, 3000, 5000, 10000, 20000],
    "TotalWorkingYears": [0, 2, 5, 10, 20, 41],
    "YearsAtCompany": [0, 1, 2, 5, 10, 41],
    "DistanceFromHome": [0, 10, 20, 30],
    "NumCompaniesWorked": [0, 1, 2, 5, 10],
}


def rate_by_bins(data, col):
    if col in CUSTOM_BINS:
        bins = pd.cut(data[col], CUSTOM_BINS[col], right=False)
    else:
        bins = pd.qcut(data[col], 5, duplicates="drop")
    return data.groupby(bins, observed=True)["left"].agg(rate="mean", n="size")


def plot_rate(ax, col):
    r = rate_by_bins(df, col)
    ax.bar(r.index.astype(str), r["rate"] * 100, color="#DD8452")
    ax.axhline(overall_rate * 100, color="black", ls="--", lw=1)
    for i, (rate, n) in enumerate(zip(r["rate"], r["n"])):
        ax.text(i, rate * 100 + 0.8, f"n={n}", ha="center", fontsize=8)
    ax.set_title(col)
    ax.set_ylabel("Attrition rate (%)")
    ax.tick_params(axis="x", rotation=30)


fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, CUSTOM_BINS):
    plot_rate(ax, col)
fig.suptitle("Attrition rate by range (dashed line = overall rate)")
fig.tight_layout()
save(fig, "03_rate_by_range")
plt.show()

In [ ]:
# ฟีเจอร์ต่อเนื่องที่เหลือ แบ่ง 5 ช่วงตามจำนวนคน
others = [c for c in CONTINUOUS if c not in CUSTOM_BINS]
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flat, others):
    plot_rate(ax, col)
for ax in axes.flat[len(others):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

---
## ขั้นตอนที่ 6: ฟีเจอร์แบบระดับ (ordinal)

ความหมายของแต่ละระดับ (เช่น 1 = Low, 4 = Very High) ดูได้ที่ [docs/dataset.md](../docs/dataset.md)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.flat, ORDINAL):
    r = df.groupby(col)["left"].agg(rate="mean", n="size")
    ax.bar(r.index.astype(str), r["rate"] * 100, color="#DD8452")
    ax.axhline(overall_rate * 100, color="black", ls="--", lw=1)
    for i, (rate, n) in enumerate(zip(r["rate"], r["n"])):
        ax.text(i, rate * 100 + 0.8, f"n={n}", ha="center", fontsize=8)
    ax.set_title(col)
    ax.set_ylabel("Attrition rate (%)")
fig.tight_layout()
save(fig, "04_ordinal_rates")
plt.show()

---
## ขั้นตอนที่ 7: ความเบ้ (skewness) และ outlier

- skew > 1 แปลว่าเบ้ขวามาก (คนส่วนใหญ่ค่าต่ำ มีไม่กี่คนค่าสูงมาก) เช่น เงินเดือน จำนวนปี โมเดลต้นไม้อย่าง XGBoost ไม่ได้รับผล แต่ Logistic Regression (baseline ใน wk2–3) ควรลอง `log1p` หรือ scale ก่อน
- outlier นับตามกฎ 1.5×IQR ในข้อมูลนี้ outlier เป็นค่าที่**เป็นไปได้จริง** (ผู้บริหารเงินเดือนสูง คนทำงานมานาน) ไม่ใช่ข้อมูลผิด จึง**ไม่ควรลบทิ้ง**

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()


shape = pd.DataFrame({
    "skew": df[NUMERIC].skew(),
    "n_outliers": df[NUMERIC].apply(iqr_outliers),
    "min": df[NUMERIC].min(),
    "max": df[NUMERIC].max(),
}).sort_values("skew", ascending=False)
shape.style.format({"skew": "{:.2f}"})

In [ ]:
# ตัวอย่าง: log1p ทำให้ MonthlyIncome เบ้น้อยลง
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["MonthlyIncome"], ax=axes[0]).set_title(f"MonthlyIncome (skew {df['MonthlyIncome'].skew():.2f})")
log_income = np.log1p(df["MonthlyIncome"])
sns.histplot(log_income, ax=axes[1]).set_title(f"log1p(MonthlyIncome) (skew {log_income.skew():.2f})")
fig.tight_layout()
plt.show()

---
## ขั้นตอนที่ 8: ความสัมพันธ์ระหว่างฟีเจอร์ (multicollinearity)

ใช้ Spearman correlation (ทนต่อความเบ้และ outlier) คู่ไหนสัมพันธ์กันสูงมาก (|r| > 0.7) แปลว่าบอกข้อมูลซ้ำกัน ไม่เป็นปัญหากับ XGBoost แต่ทำให้:
- ค่าสัมประสิทธิ์ของ Logistic Regression แกว่งและตีความยาก
- ค่า SHAP ถูก "แบ่ง" ระหว่างฟีเจอร์ที่ซ้ำกัน (เกี่ยวกับงาน SHAP ใน wk4–5) เช่น ความสำคัญของ "ความอาวุโส" อาจกระจายไปทั้ง JobLevel, MonthlyIncome, TotalWorkingYears

In [ ]:
corr = df[NUMERIC].corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            annot=True, fmt=".1f", annot_kws={"size": 7}, ax=ax)
ax.set_title("Spearman correlation between numeric features")
save(fig, "05_correlation")
plt.show()

In [ ]:
pairs = corr.where(~mask).stack().rename("spearman_r").reset_index()
pairs.columns = ["feature_a", "feature_b", "spearman_r"]
pairs[pairs["spearman_r"].abs() > 0.7].sort_values("spearman_r", ascending=False)

In [ ]:
# correlation ของแต่ละฟีเจอร์กับการลาออกโดยตรง (point-biserial = Pearson กับ target 0/1)
corr_target = df[NUMERIC].corrwith(df["left"]).sort_values()
fig, ax = plt.subplots(figsize=(7, 8))
corr_target.plot.barh(ax=ax, color=["#4C72B0" if v < 0 else "#DD8452" for v in corr_target])
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Correlation with attrition (point-biserial)")
plt.show()

---
## ขั้นตอนที่ 9: สรุปผล (ฟีเจอร์ตัวเลขที่น่าสนใจที่สุด)

ตัวเลขด้านล่างมาจากข้อมูลดิบ 1,470 คน (อัตราลาออกรวม 16.1%) ถ้ารันแล้วได้ไม่ตรง ให้เช็คว่าใช้ไฟล์ `data/raw/` ตัวเดียวกัน

1. **ประสบการณ์และอายุงานน้อย = ลาออกมาก** เป็นกลุ่มที่แยกได้ชัดที่สุด (effect ~ -0.30) `TotalWorkingYears` มัธยฐานกลุ่มลาออก 7 ปี เทียบกับ 10 ปี คนที่ทำงานมาไม่ถึง 2 ปีลาออก **48.9%** (3 เท่าของค่าเฉลี่ย) ส่วน `YearsAtCompany` ปีแรก ๆ (< 2 ปี) ลาออก ~35% แต่หลัง 5 ปีลดเหลือ ~11%
2. **รายได้ต่ำ = ลาออกมาก** `MonthlyIncome` มัธยฐาน 3,202 vs 5,204 (effect -0.31) คนรายได้ต่ำกว่า 3,000 ลาออก **28.6%** เทียบกับ 8.9% ของคนรายได้เกิน 10,000 ไปทางเดียวกับ `JobLevel` ระดับ 1 ที่ลาออก 26.3%
3. **อายุน้อย = ลาออกมาก** อายุต่ำกว่า 25 ปีลาออก **39.2%** อายุ 35–45 ปีเหลือ 10.1%
4. **ไม่มี stock option ลาออกมากกว่า** `StockOptionLevel = 0` ลาออก 24.4% เทียบกับ 7.6–9.4% ของระดับ 1–2 (ใน context ไทยเทียบได้กับกองทุนสำรองเลี้ยงชีพ ตาม Localization Notes ใน README) ฟีเจอร์ความพึงพอใจ/การมีส่วนร่วม ระดับต่ำสุด (1) ลาออกสูงทุกตัว เช่น `JobInvolvement = 1` 33.7%, `WorkLifeBalance = 1` 31.2%, `EnvironmentSatisfaction = 1` 25.4% แต่ระดับ 2–4 ต่างกันไม่มาก
5. **ฟีเจอร์ที่แทบไม่ช่วย** `HourlyRate`, `DailyRate`, `MonthlyRate`, `PercentSalaryHike`, `PerformanceRating`, `Education` มี |effect| < 0.1 (DailyRate/HourlyRate/MonthlyRate ไม่สัมพันธ์กับ MonthlyIncome ด้วย น่าจะเป็นค่าสุ่มของ dataset จำลอง) ส่วน `DistanceFromHome` มีผลเล็กน้อย (ไกล 20+ ลาออก 21.4% vs 14.1% ของคนที่ใกล้กว่า 10) ซึ่งทีมเอาไปทำฟีเจอร์ `OverTimeXDistance` แล้ว

### ข้อควรระวังสำหรับการเทรนโมเดล
- ฟีเจอร์ "ความอาวุโส" สัมพันธ์กันเองสูง: `JobLevel`–`MonthlyIncome` (r ≈ 0.95), `TotalWorkingYears`–`MonthlyIncome`/`JobLevel` (≈ 0.77), กลุ่ม `YearsAtCompany`/`YearsInCurrentRole`/`YearsWithCurrManager` (≈ 0.76) และ `PercentSalaryHike`–`PerformanceRating` (≈ 0.77) ตอนตีความ SHAP หรือ coefficient ต้องดูเป็นกลุ่ม
- `MonthlyIncome` และฟีเจอร์ `Years*` เบ้ขวา (skew 1–2) ถ้าใช้ Logistic Regression ควร `log1p` / scale ก่อน
- outlier เป็นค่าที่เป็นไปได้จริง ไม่ลบทิ้ง
- ทั้งหมดนี้เป็น **ความสัมพันธ์ ไม่ใช่เหตุและผล** และเป็นข้อมูลจำลองของ IBM (ไม่ใช่บริษัทไทยจริง) สกุลเงินของ `MonthlyIncome` ไม่ระบุ (ดู DE-01)